### Module's Run Time

**Plain Language First**

Python doesn't execute your entire program at once. Different constructs "wake up" at different moments. The confusion comes from assuming everything runs when you call it — but a lot happens before any function is ever called.

One sentence per "time":
- **Import time**       → Python runs the entire file top-to-bottom the moment you import it
- **Class body time**   → the class body executes immediately when Python reaches the `class` statement — before any instance exists
- **Function def time** → `def` creates a function object and freezes the default args — the body is NOT run yet
- **Decorator time**    → the decorator is applied right at `def` time, not when the function is later called
- **Function call time** → only NOW does the function body actually execute
- **Instance time**     → `__new__` then `__init__` run when you call `MyClass()`

---

### The Timeline on One File
```py
print("A — module top level")           # runs at import time

class MyClass:                           # class body runs NOW
    print("B — class body")             # runs at class body time

    x = 10                              # evaluated at class body time

    def method(self):                   # function object created NOW
        print("C — method body")        # NOT run yet

print("D — still top level")           # runs at import time

obj = MyClass()                         # instance time → __init__
obj.method()                            # function call time → body runs
```
```shell
Output when this file is imported:
  A — module top level
  B — class body          ← class body ran, no instance yet!
  D — still top level

Output when obj.method() is called:
  C — method body
```
`B` printed before any instance was ever created.

That surprises most developers the first time.

---

### Each "Time" in Detail
**1. Import Time**
```py
# config.py
print("config.py is being imported")    # runs HERE

DB_URL = os.environ.get("DATABASE_URL") # evaluated HERE

class Settings:                         # class body runs HERE
    debug = True
```

```py
# main.py
import config   # ← everything above runs RIGHT NOW
                #   "config.py is being imported" prints immediately
                #   DB_URL is read from env immediately
```

**Key facts:**
- Python runs the entire module top-to-bottom on first import
- Second import of the same module → hits sys.modules cache, nothing re-runs
- This is why heavy imports slow startup — code runs at import time
- circular imports fail because the module isn't finished importing yet

---

**2. Class Body Execution Time**

The class body is **just a code block**. Python executes it **immediately**, collects the namespace, then builds the class object.
```py
class Order:
    print("class body running")         # runs NOW, at class definition

    # These all run at class body time:
    MAX_ITEMS   = 100                   # class variable created NOW
    item_count  = len([1, 2, 3])        # expression evaluated NOW

    status_map  = {                     # dict created NOW
        "pending":   0,
        "completed": 1,
    }

    def __init__(self):                 # function object created NOW
        pass                            # body not run yet

# __set_name__ is called HERE (class body time) — that's why
# descriptors know their own name before any instance is created
```
**Everything at class body time**:
- Class variables assigned
- Method function objects created (not called)
- `__set_name__` called on descriptors
- `__init_subclass__` called on the parent class
- Metaclass `__new__` and `__init__` called

---

**3. Function Definition Time**
```py
import time

def slow_default(
    data: list = [],                         # ← list created ONCE, here
    timestamp: float = time.time(),          # ← time.time() called ONCE, here
    conn = get_db_connection(),              # ← DB connection opened ONCE, here!
):
    ...
```
**At def time**:
- The function **object** is created
- ALL **default argument** expressions are evaluated — exactly once
- The **decorator**(s) are applied (see next section)
- The function body is **stored** but NOT run
- Type annotations are **evaluated** (unless `from __future__ import annotations`)

**At call time**:
- Arguments are **bound to** parameters
- The function **body runs**
- Local variables are created **fresh each call**

---

**4. Decorator Application Time**
```py
def log(func):
    print(f"decorating {func.__name__}")   # runs at DEFINITION time
    def wrapper(*args, **kwargs):
        print(f"calling {func.__name__}")  # runs at CALL time
        return func(*args, **kwargs)
    return wrapper

@log
def greet(name):                 # log(greet) is called HERE — at def time
    print(f"Hello {name}")
```
**Output at definition time** (before `greet` is ever called):
```shell
"decorating greet"
```

**Output when `greet("Alice")` is called**:
```shell
  "calling greet"
  "Hello Alice"
```

**This means**:
- `@router.get("/users")` registers the route at **import time**
- `@pytest.mark.parametrize` wraps the test at **collection time**
- `@property` creates the descriptor object at **class body time**
- A slow decorator slows startup, not requests

In [ ]:
def log(func):
    print(f"decorating {func.__name__}")   # runs at func's DEFINITION time when (log(func)) is CALLED
    def wrapper(*args, **kwargs):          # which happens at greet's def time
        print(f"calling {func.__name__}")  # runs at CALL time
        return func(*args, **kwargs)
    return wrapper

@log                             # greet = log(greet)
def greet(name):                 # log(greet) is called HERE — at greet's definition time
    print(f"Hello {name}")

decorating greet


1. **`log`'s def time**:
    - `log` function object created, body stored
    - `"decorating..."` line is NOT run yet

2. **`greet`'s def time**:
    - Python calls `log(greet)` to apply the decorator
    - `log`'s body NOW runs → `"decorating greet"` prints
    - `wrapper` function object is created and returned
    - the name `greet` now points to `wrapper`

3. **`greet`'s call time**:
    - `wrapper` body runs (actually `wrapper`)
    - "calling greet" prints

### The Gotcha Map — Where Time Causes Bugs
**Gotcha 1** — Mutable Default Argument

In [ ]:
# ❌ list created ONCE at def time — shared across all calls
def add_item(item, cart=[]):
    cart.append(item)
    return cart

add_item("apple")   # ["apple"]
add_item("bread")   # ["apple", "bread"] ← same list!

# ✅ None sentinel — new list created at call time
def add_item(item, cart=None):
    if cart is None:        # call time — fresh list every call
        cart = []
    cart.append(item)
    return cart

**Gotcha 2** — Late Binding Closures

In [ ]:
# ❌ i is looked up at CALL time, not def time
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])    # [2, 2, 2] — i=2 by the time any are called

# ✅ default arg captures i at DEF time (each lambda's def time)
funcs = [lambda i=i: i for i in range(3)]
print([f() for f in funcs])    # [0, 1, 2]

**Gotcha 3** — Class Variable Shared Across Instances

In [ ]:
class User:
    history = []               # created ONCE at class body time — SHARED

u1 = User(); u2 = User()
u1.history.append("login")
print(u2.history)              # ["login"] ← shared!

class User:
    def __init__(self):
        self.history = []      # created at instance time — each gets their own

**Gotcha 4** — Type Annotations Evaluated at Def Time

In [ ]:
# ❌ Forward reference fails — Bar not defined yet at def time
class Foo:
    def get_bar(self) -> Bar:   # NameError: Bar not defined!
        ...

class Bar:
    pass

# ✅ Option 1: string annotation (deferred)
class Foo:
    def get_bar(self) -> "Bar": ...

# ✅ Option 2: modern (3.10+) — all annotations become lazy strings
from __future__ import annotations

class Foo:
    def get_bar(self) -> Bar: ...  # not evaluated at def time anymore

### The Full Picture
```shell
Source file touched:

  PARSE TIME        Python reads file → builds AST (abstract syntax tree)
  COMPILE TIME      AST → bytecode (.pyc file)

File imported / run:

  IMPORT TIME       top-level code executes top-to-bottom
    │
    ├── CLASS BODY TIME    class statement reached → body executes
    │     │
    │     ├── __set_name__ called on descriptors
    │     ├── __init_subclass__ called on parent
    │     └── metaclass __new__ / __init__ called
    │
    ├── DEF TIME           def statement reached → function object created
    │     │
    │     ├── default args evaluated NOW
    │     ├── annotations evaluated NOW (unless __future__)
    │     └── decorator applied NOW
    │
    └── CALL TIME          func() encountered → body runs
          │
          ├── arguments bound
          ├── locals created fresh
          └── INSTANCE TIME when MyClass() called
                ├── __new__ (creates object)
                └── __init__ (initialises it)
```

### Interview Angle
"What's the difference between definition time and call time?"
  → Interviewers ask this to test whether you understand
    why mutable default arguments are a bug, not a feature.
    The answer is the entire section above.

"When does a decorator run?"
  → At definition time. The wrapper runs at call time.
    Knowing this explains why @app.route() registers routes
    at startup, not per-request.

"Why does this print before any instance is created?"
  class Foo:
      print("hello")      # ← this
  → Because the class body is executed at class definition time,
    not at instantiation time.

"When are default arguments evaluated?"
  → Once, at function definition time.
    This is the answer to the mutable default argument gotcha
    and the timestamp=time.time() gotcha.

> The mental model that ties it all together: Python executes code as it reads it, top to bottom. def and class are executable statements — but def stores the body for later, while class runs the body immediately.